In [1]:
from torch import nn
import torch
from torch import tensor
import pandas as pd
import stanza
from collections import Counter
from torch.optim.lr_scheduler import OneCycleLR
import re
import nltk
from nltk import Tree

In [4]:
%env LD_LIBRARY_PATH=

env: LD_LIBRARY_PATH=


In [5]:
stanza.download('en')
nlp = stanza.Pipeline(lang='en', processors='tokenize,pos,constituency')

2026-07-31 10:04:52 INFO: Downloaded file to /home/klw66/.cache/stanza/1.13.0/resources/resources.json
2026-07-31 10:04:52 INFO: Downloading default packages for language: en (English) ...
2026-07-31 10:04:52 INFO: File exists: /home/klw66/.cache/stanza/1.13.0/resources/en/default.zip
2026-07-31 10:04:54 INFO: Finished downloading models and saved to /home/klw66/.cache/stanza/1.13.0/resources
2026-07-31 10:04:54 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-07-31 10:04:55 INFO: Downloaded file to /home/klw66/.cache/stanza/1.13.0/resources/resources.json
2026-07-31 10:04:55 WARNING: Language en package default expects mwt, which has been added
2026-07-31 10:04:55 INFO: Loading these models for language: en (English):
| Processor    | Package             |
--------------------------------------
| tokenize     | combined_nocharlm   |
| mwt          | combined            |
| pos          | combined_charlm     |
| constituency | ptb3-revised_charlm |

2026-07-31 10:04:55 INFO: Using device: cuda
2026-07-31 10:04:55 INFO: Loading: tokenize
2026-07-31 10:04:57 INFO: Loading: mwt
2026-07-31 10:04:57 INFO: Loading: pos
2026-07-31 10:04:58 INFO: Loading: constituency
2026-07-31 10:04:58 INFO: Done loading processors!


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
def format_vectors(vector):
    clean = re.sub(r",?\s*grad_fn=<[^>]+>", "", vector)
    return (eval(clean), {"tensor": torch.tensor})[0]

In [8]:
df = pd.read_csv("pass_quest_H.csv", engine="python", encoding='utf-8')
df['context'] = df['context'].apply(format_vectors)

In [9]:
model_size = df['context'].iloc[0].shape[1]
print(model_size)
print(len(df))

256
9960


In [27]:
vocab = {}
roles_dict = {}

def tree_to_list(tree):
    if tree.label() == 'ROOT' and len(tree) == 1:
        tree = tree[0]

    return _collapse(tree)


def _collapse(node):
    children = list(node)  # nltk.Tree is list-like; this gives child nodes/strings

    if len(children) == 1:
        child = children[0]
        if isinstance(child, str):  # leaf check: NLTK leaves are plain strings
            try:
                fill_token = torch.tensor(vocab[child], dtype=torch.long)
            except KeyError:
                fill_token = torch.tensor(len(vocab), dtype=torch.long)
                vocab[child] = len(vocab)
            try:
                role_token = torch.tensor(roles_dict[node.label()], dtype=torch.long)
            except KeyError:
                role_token = torch.tensor(len(roles_dict), dtype=torch.long)
                roles_dict[node.label()] = len(roles_dict)

            return [fill_token, role_token]

        collapsed_child = _collapse(child)
        try:
            role_token = torch.tensor(roles_dict[node.label()], dtype=torch.long)
        except KeyError:
            role_token = torch.tensor(len(roles_dict), dtype=torch.long)
            roles_dict[node.label()] = len(roles_dict)

        return [collapsed_child[0], role_token]

    try:
        role_token = torch.tensor(roles_dict[node.label()], dtype=torch.long)
    except KeyError:
        role_token = torch.tensor(len(roles_dict), dtype=torch.long)
        roles_dict[node.label()] = len(roles_dict)
    return [[_collapse(c) for c in children], role_token]

def test_bin(tree):
    if not isinstance(tree, nltk.Tree):
        return False  # it's a leaf string, no children to count
    if len(tree) > 2:
        return True
    return any(test_bin(child) for child in tree)

In [28]:
import re

_CONDENSE_PATTERN = re.compile(r'^NP\|<(?:JJ-)+(?:NN|NNS)>$')

def conservative_condense(tree: Tree) -> Tree:
    """
    Given an nltk Tree in Chomsky Normal Form, mutate it in place (and return it) so that:

    1. A VBN preterminal is relabeled JJ if the *next* preterminal in linear
       (left-to-right) order is NN or JJ.
    2. Any node whose label matches NP|<JJ-JJ-...-NN(S)?> (one or more JJs
       followed by NN or NNS) is relabeled to "NP|JN".
    """
    # --- Step 1: VBN -> JJ before NN/JJ ---
    # Preterminals (POS-tag nodes directly over a single leaf) have height 2.
    # tree.subtrees() is a pre-order DFS, so restricting to height==2 nodes
    # yields them in left-to-right sentence order.
    preterminals = list(tree.subtrees(lambda t: t.height() == 2))

    for i, pt in enumerate(preterminals):
        if pt.label() == 'VBN' and i + 1 < len(preterminals):
            next_label = preterminals[i + 1].label()
            if next_label in ('NN', 'JJ'):
                pt.set_label('JJ')

    # --- Step 2: condense NP|<JJ-...-NN(S)> labels ---
    for st in tree.subtrees():
        if _CONDENSE_PATTERN.match(st.label()):
            st.set_label('NP|JN')

    return tree

def hard_condense(tree: Tree) -> Tree:
    """
    Removes the artificial CNF markup from labels, turning e.g.
    "NP|<JJ-NN>", "VP|<NP-NP>", "NP|<JJ-JJ-JJ-NN>" into just "NP", "VP", "NP".
    Keeps the tree structure (including the extra binarization nesting) intact
    -- only the labels change. Mutates and returns the tree.
    """
    for st in tree.subtrees():
        label = st.label()
        if '|' in label:
            st.set_label(label.split('|')[0])
    return tree

In [29]:
import re


def clean_sentence(s):
    s = s.strip()

    # Remove a trailing "active" along with the period right before it
    s = re.sub(r'\.\s*active\s*$', '', s)

    # Remove a trailing "passive" along with the period right before it
    s = re.sub(r'\.\s*passive\s*$', '', s)

    # Underscores -> spaces
    s = s.replace('_', ' ')

    # Collapse any double spaces left behind and trim
    s = re.sub(r'\s+', ' ', s).strip()

    return s

In [30]:
all_src = []

for i in df["input"].tolist():
    doc = nlp(clean_sentence(i.lower()))
    stanza_tree = doc.sentences[0].constituency
    nltk_tree = Tree.fromstring(str(stanza_tree))
    nltk_tree.chomsky_normal_form()
    nltk_tree = conservative_condense(nltk_tree)
    
    tree = tree_to_list(nltk_tree)
    is_not_binary = test_bin(nltk_tree)
    if not is_not_binary:
        all_src.append(tree[0])

In [31]:
print(len(all_src))

9960


In [32]:
import json

df["treed"] = all_src
df.to_csv("10k_conservative_pq_w_tree.csv", index = False)

with open("vocab_dict.json", "w") as file:
    json.dump(vocab, file, indent=4)

with open("roles_dict.json", "w") as file:
    json.dump(roles_dict, file, indent=4)

In [13]:
import json

tree_df = pd.read_csv("pq_w_tree.csv")
all_src = tree_df["treed"].tolist()
all_src = [eval(x) for x in all_src]

with open('vocab_dict.json', 'r') as file:
    vocab = json.load(file)

with open('roles_dict.json', 'r') as file:
    roles_dict = json.load(file)

In [33]:
vocab_size = len(vocab)
role_size = len(roles_dict)
print(vocab_size)
print(role_size)

63
185


In [34]:
from sklearn.model_selection import train_test_split


all_trg = df["context"].tolist()
random_state=1
# First split: 80% train, 20% temp (which becomes val+test)
train_src, temp_src, train_trg, temp_trg = train_test_split(
    all_src, all_trg, test_size=0.2, random_state = random_state
)

# Second split: split the 20% temp into 10% val, 10% test
val_src, test_src, val_trg, test_trg = train_test_split(
    temp_src, temp_trg, test_size=0.5, random_state = random_state
)

print(len(train_src), len(val_src), len(test_src))

7968 996 996


In [44]:
class RTPREncode(nn.Module):
    def __init__(self, nfillers, nroles, filler_size, role_size, model_size):
        super(RTPREncode, self).__init__()
        self.fillers = nn.Embedding(nfillers, filler_size)
        self.roles = nn.Embedding(nroles, role_size)
        self.filler_size = filler_size
        self.role_size = role_size
        self.model_size = model_size
        self.dropout = nn.Dropout(0.4)
        self.w = torch.nn.Parameter(torch.randn(filler_size, filler_size*role_size)*0.01)
        self.wo = torch.nn.Parameter(torch.randn(model_size, filler_size)*0.01)

    def forward(self, x):
        result = torch.zeros(self.filler_size, self.role_size, device=self.w.device)
        for pair in x:
            # print(pair)
            filler = pair[0]
            role = pair[1].to(device)
            
            if isinstance(filler, list):
                filler = self(filler)
            else:
                filler = self.fillers(filler.to(device))
                # filler = self.dropout(filler)
            
            role = self.roles(role)
            # role = self.dropout(role)
            result += torch.outer(filler, role)
        
        result = torch.flatten(result)
        result = self.w @ result
        return result

In [71]:
class TwoWRTPREncode(nn.Module):
    def __init__(self, nfillers, nroles, filler_size, role_size, model_size):
        super(TwoWRTPREncode, self).__init__()
        self.fillers = nn.Embedding(nfillers, filler_size)
        self.roles = nn.Embedding(nroles, role_size)
        self.filler_size = filler_size
        self.role_size = role_size
        self.model_size = model_size
        # self.dropout = nn.Dropout(0.4)
        self.wL = torch.nn.Parameter(torch.randn(filler_size, filler_size*role_size)*0.01)
        self.wR = torch.nn.Parameter(torch.randn(filler_size, filler_size*role_size)*0.01)
        self.wo = torch.nn.Parameter(torch.randn(model_size, filler_size)*0.01)

    def forward(self, x):
        result = torch.zeros(self.filler_size, device=device)
        #ONLY WORKS HERE BECAUSE BINARY!!
        for i in range(2):
            pair = x[i]
            filler = pair[0]
            role = pair[1].to(device)
            
            if isinstance(filler, list):
                filler = self(filler)
            else:
                filler = self.fillers(filler.to(device))
            
            role = self.roles(role)
            binding = torch.outer(filler, role)
            if i == 0:
                weight_res = self.wL @ torch.flatten(binding)
            elif i == 1:
                weight_res = self.wR @ torch.flatten(binding)
            result += (weight_res)
        
        return result

In [39]:
class TwoRRTPREncode(nn.Module):
    def __init__(self, nfillers, num_ws, filler_size, role_size, model_size):
        super(TwoRRTPREncode, self).__init__()
        self.fillers = nn.Embedding(nfillers, filler_size)
        self.roles = nn.Embedding(2, role_size)
        self.filler_size = filler_size
        self.role_size = role_size
        self.model_size = model_size
        # self.dropout = nn.Dropout(0.4)
        self.w_list = torch.nn.ParameterList([
            torch.nn.Parameter(torch.randn(filler_size, filler_size * role_size) * 0.01)
            for _ in range(num_ws)
        ])
        self.wo = torch.nn.Parameter(torch.randn(model_size, filler_size)*0.01)

    def forward(self, x):
        result = torch.zeros(self.filler_size, self.role_size, device=device)
        for i in range(2):
            # print(pair)
            pair = x[i]
            filler = pair[0]
            syntactic_label = pair[1].to(device)
            
            if isinstance(filler, list):
                filler = self(filler)
            else:
                filler = self.fillers(filler.to(device))
            
            #left or right
            role = self.roles(torch.tensor(i, dtype=torch.long).to(device))
            
            result += torch.outer(filler, role)
        
        result = torch.flatten(result)

        
        result = self.w_list[syntactic_label.item()] @ result
        return result

In [52]:
class BobRTPREncode(nn.Module):
    def __init__(self, nfillers, nroles, filler_role_size, lr_cat_word_size, model_size):
        super(BobRTPREncode, self).__init__()
        self.fillers = nn.Embedding(nfillers, filler_role_size)
        self.synt_labels = nn.Embedding(nroles, filler_role_size)
        self.filler_role_size = filler_role_size
        self.lr_cat_word_size = lr_cat_word_size
        self.model_size = model_size
        # self.dropout = nn.Dropout(0.4)
        self.cat = torch.nn.Parameter(torch.randn(lr_cat_word_size)*0.01)
        self.word = torch.nn.Parameter(torch.randn(lr_cat_word_size)*0.01)
        self.left = torch.nn.Parameter(torch.randn(lr_cat_word_size)*0.01)
        self.right = torch.nn.Parameter(torch.randn(lr_cat_word_size)*0.01)

        self.w_res = torch.nn.Parameter(torch.randn(filler_role_size, filler_role_size*lr_cat_word_size)*0.01)
        self.w_cw = torch.nn.Parameter(torch.randn(filler_role_size, filler_role_size*lr_cat_word_size)*0.01)
        self.wo = torch.nn.Parameter(torch.randn(model_size, filler_role_size)*0.01)

    def forward(self, x):
        result = torch.zeros(self.lr_cat_word_size, self.filler_role_size, device=self.w_res.device)
        for i in range(2):
            pair = x[i]
            filler = pair[0]
            synt_label = pair[1].to(device)
            
            if isinstance(filler, list):
                filler = self(filler)
            else:
                filler = self.fillers(filler.to(device))
            
            synt_label = self.synt_labels(synt_label)

            cat_binding = torch.outer(self.cat, synt_label)
            word_binding = torch.outer(self.word, filler)

            pair_res = self.w_cw @ torch.flatten(cat_binding + word_binding)

            if i == 0:
                lr_res = torch.outer(self.left, pair_res)
            elif i == 1:
                lr_res = torch.outer(self.right, pair_res)

            result += lr_res
        
        result = torch.flatten(result)
        result = self.w_res @ result
        return result

In [53]:
#Evaluate a model (for validation and testing loss)
def evaluate(model, src, trg, criterion, device):
    with torch.no_grad():
        model.eval()
        epoch_loss = 0.0

        for i in range(len(src)):
            src_batch = src[i]
            trg_batch = trg[i].float().to(device)
            encoding = model(src_batch)
            output = torch.matmul(encoding, model.wo.T)    # (batch_size, model_size)
            loss = criterion(output, trg_batch.squeeze(0))
            epoch_loss += loss.item()

        return epoch_loss / len(src)

In [54]:
# Train with optimized hyperparams
num_epochs = 25

model = BobRTPREncode(vocab_size, role_size, 64, 64, model_size).to(device)
learning_rate = 1e-4
norm_val = 5
max_rate = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
criterion = nn.MSELoss()
scheduler = OneCycleLR(optimizer, max_lr=max_rate, steps_per_epoch=len(train_src), epochs=num_epochs)

In [ ]:
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1} / {num_epochs}")
    # print(f"Trial {trial.number}")
    model.train()

    epoch_loss = 0.0
    with torch.set_grad_enabled(True):
        for i in range(len(train_src)):
            src_batch = train_src[i]
            trg_batch = train_trg[i].float().to(device)
            optimizer.zero_grad()
            try:
                encoding = model(src_batch)
            except TypeError:
                print(src_batch)
                print(i)
            except IndexError:
                print(src_batch)
                print(i)
            output = torch.matmul(encoding, model.wo.T)

            loss = criterion(output, trg_batch.squeeze(0))
            loss.backward()

            #Norm=magnitude, clipping gradient to ensure they don't explode
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = norm_val)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

    print(f"Train Loss: {epoch_loss/len(train_src)}")
    val_loss = evaluate(model, val_src, val_trg, criterion, device)
    print(f"Validation Loss: {val_loss}")

Epoch 1 / 25
Train Loss: 0.17262355398843027
Validation Loss: 0.16239006595917974
Epoch 2 / 25
Train Loss: 0.16284662689059884
Validation Loss: 0.16312610891359638
Epoch 3 / 25
Train Loss: 0.16341810542005433
Validation Loss: 0.16353491734129838
Epoch 4 / 25
Train Loss: 0.16358677885050035
Validation Loss: 0.163395794048666
Epoch 5 / 25


In [ ]:
print(f"Train Loss: {evaluate(model, train_src, train_trg, criterion, device)}")
print(f"Val Loss: {evaluate(model, val_src, val_trg, criterion, device)}")
print(f"Test Loss: {evaluate(model, test_src, test_trg, criterion, device)}")

In [ ]:
# model.load_state_dict(torch.load("new_100e_twor_pqH_synt_tpr.pt"))
torch.save(model.state_dict(), "new_100e_twor_pqH_synt_tpr.pt")

In [ ]:
def objective(trial):
    # Train with optimized hyperparams
    num_epochs = 30
    
    model = TwoRRTPREncode(vocab_size, role_size, 64, 64, model_size).to(device)
    learning_rate = trial.suggest_float("lr", 1e-5, 1e-3, log = True)
    norm_val = trial.suggest_float("grad_norm", 1, 10, log = True)
    max_rate = trial.suggest_float("scheduler_max", 1e-4, 1e-2, log = True)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    criterion = nn.MSELoss()
    scheduler = OneCycleLR(optimizer, max_lr=max_rate, steps_per_epoch=len(train_src), epochs=num_epochs)
    
    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1} / {num_epochs}")
        # print(f"Trial {trial.number}")
        model.train()

        epoch_loss = 0.0
        with torch.set_grad_enabled(True):
            for i in range(len(train_src)):
                src_batch = train_src[i]
                trg_batch = train_trg[i].float().to(device)
                optimizer.zero_grad()

                encoding = model(src_batch)
                output = torch.matmul(encoding, model.wo.T)

                loss = criterion(output, trg_batch.squeeze(0))
                loss.backward()

                #Norm=magnitude, clipping gradient to ensure they don't explode
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = norm_val)
                optimizer.step()
                scheduler.step()
                epoch_loss += loss.item()

        print(f"Train Loss: {epoch_loss/len(train_src)}")
        val_loss = evaluate(model, val_src, val_trg, criterion, device)

        trial.report(val_loss, epoch)
        #prune cuts off trial early
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        print(f"Validation Loss: {val_loss}")

    return val_loss

In [ ]:
import optuna 

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=8)
print("Best Hyperparameters:", study.best_params) 

In [ ]:
pruned_trials = study.get_trials(deepcopy=False, states=[optuna.trial.TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[optuna.trial.TrialState.COMPLETE])

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

In [ ]:
final_df = pd.read_csv("pq_w_tree.csv")

concat_srcs = train_src + val_src + test_src
concat_trgs = train_trg + val_trg + test_trg

all_source = []
all_contexts = []

model.eval()
with torch.no_grad():
    for src, trg in zip(concat_srcs, concat_trgs):
        encoding = model(src)
        output = torch.matmul(encoding, model.wo.T)
        all_contexts.append(output)
        all_source.append(src)

In [ ]:
def is_tensor(x):
    """Duck-type check: works for torch.Tensor or anything with .item()."""
    return hasattr(x, "item") and not isinstance(x, list)

def get_keys(node, reverse_lookup, results=None):
    """
    Recursively walk `node`. Whenever a [tensor, tensor] leaf pair is found,
    take the left tensor's .item() value, use it to find the matching key
    in `reverse_lookup` (built from your original dict), and append it
    to a flat list (left-to-right order).
    """
    if results is None:
        results = []

    if isinstance(node, list):
        if len(node) == 2 and is_tensor(node[0]) and is_tensor(node[1]):
            value = node[0].item()
            results.append(reverse_lookup.get(value))
        else:
            for child in node:
                get_keys(child, reverse_lookup, results)

    return results

def detokenize(words):
    sentence = ""
    prev_word = None

    for word in words:
        # Special edge case: "high" followed by "five" -> "high_five"
        if prev_word == "high" and word == "five":
            # Remove the " high" (or "high") already added, replace with "high_five"
            sentence = sentence[: -len("high")] + "high_five"
            prev_word = "high_five"
            continue

        if not sentence:
            sentence = word
            prev_word = word
            continue

        # No space before punctuation or contraction pieces (n't, 's, 're, 've, 'll, 'd, 'm)
        if re.match(r"^(n't|'s|'re|'ve|'ll|'d|'m)$", word) or word in {".", ",", "!", "?", ";", ":"}:
            sentence += word
        else:
            sentence += " " + word

        prev_word = word

    return sentence

In [ ]:
reverse_vocab = {v: k for k, v in vocab.items()}

In [ ]:
new_all_src = []

for i in all_source:
    new_all_src.append(detokenize(get_keys(i, reverse_vocab)))

In [ ]:
reconstructed_sources = []
reconstructed_targets = []

for i in new_all_src:
    val = re.sub(r'\.\s*passive\s*$', '', i)
    mask = df['input'].str.contains(val, case=False)
    matching_indices = df[mask].index
    try:
        reconstructed_targets.append(df.loc[matching_indices, 'target'].values[0])
        reconstructed_sources.append(df.loc[matching_indices, 'input'].values[0])
    except IndexError:
        print(i)

In [ ]:
context_df = pd.DataFrame({
        "source": reconstructed_sources,
        "TPR out": all_contexts,
        "target": reconstructed_targets
    })

context_df.to_csv("twor_prelim_results.csv", index=False)